# Salinas Spatial Split — MorphFormer (Official)

This notebook adapts the **official `mhaut/morphFormer` implementation** to the project's Salinas spatial split. The official `CNN` class is retained; inputs are formatted as the official implementation expects: **`[batch, spectral_bands, patch_size × patch_size]`**. The model reshapes the flattened patches internally.

The notebook uses the project's existing spatial split (`seed42`) and the same benchmark conditions as the other model notebooks:
- Training batch size: 128 (as configured in the supplied MorphFormer notebook)
- Benchmark batch size: 128
- Warm-up: 20 forward passes
- Timed inference: 100 forward passes
- Metrics: accuracy, macro-F1, parameters, checkpoint size, FLOPs when THOP supports the model, peak GPU memory, latency distribution, per-sample latency and throughput.

The notebook deliberately does **not** run THOP on the model used for latency measurements. FLOPs profiling uses a deep copy, and a profiling failure is recorded as unavailable rather than stopping the rest of the benchmark.

In [1]:
from pathlib import Path
import sys
import os
import time
import copy
import subprocess
import urllib.request
import gc
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Locate the project root without changing the working directory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.name == "notebooks" and (PROJECT_ROOT.parent / "src").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    elif (PROJECT_ROOT.parent / "src").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        "Could not locate project root (expected a 'src' directory). "
        "Set PROJECT_ROOT manually to the repository root."
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dfm.data.salinas import SalinasPatchDataset, load_salinas

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
outputs_dir = PROJECT_ROOT / "outputs" / "salinas"
outputs_dir.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Project root:", PROJECT_ROOT)
print("Device:", device)
print("Outputs:", outputs_dir)

c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup
Device: cuda
Outputs: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas


In [2]:
# Load the project's fixed spatial split.
split_path = outputs_dir / "salinas_spatial_split_seed42.npz"
if not split_path.exists():
    raise FileNotFoundError(
        f"Spatial split file not found: {split_path}\n"
        "Create/use the project's seed42 spatial split before running this notebook."
    )

split = np.load(split_path)
train_indices = split["train_indices"]
val_indices = split["val_indices"]
test_indices = split["test_indices"]

print("Spatial split:", split_path.name)
print("Train / validation / test:", len(train_indices), len(val_indices), len(test_indices))

Spatial split: salinas_spatial_split_seed42.npz
Train / validation / test: 32337 10952 10840


In [3]:
# Load Salinas using the project's existing data loader.
data_dir = PROJECT_ROOT / "data" / "raw" / "salinas"
scene = load_salinas(data_dir, download=True)

print("Cube (H, W, C):", scene.cube.shape)
print("Labels:", scene.labels.shape)
print("Spectral bands:", scene.bands)
print("Classes in scene metadata:", len(scene.class_names))

Cube (H, W, C): (512, 217, 204)
Labels: (512, 217)
Spectral bands: 204
Classes in scene metadata: 16


## Official MorphFormer implementation

The repository is [mhaut/morphFormer](https://github.com/mhaut/morphFormer). Its `model.py` defines `CNN(FM, NC, Classes, HSIOnly)` and uses a module-level `patchsize`. The official `forward` reshapes each input to `[B, NC, patchsize, patchsize]`; the original training script flattens each patch to `[B, NC, patchsize × patchsize]` before passing it to the model. This notebook follows that input convention.

In [4]:
# Locate a valid checkout of the official MorphFormer repository.
# If the dedicated checkout directory is incomplete, recreate only that directory.
external_dir = PROJECT_ROOT / "external"
external_dir.mkdir(parents=True, exist_ok=True)
repo_dir = external_dir / "morphFormer"
repo_url = "https://github.com/mhaut/morphFormer.git"

model_file = None
if repo_dir.exists():
    candidates = list(repo_dir.rglob("model.py"))
    if candidates:
        # Prefer model.py directly under the repository root; otherwise use the
        # nested repository layout if one exists.
        model_file = next(
            (candidate for candidate in candidates if candidate.parent == repo_dir),
            candidates[0],
        )
        repo_dir = model_file.parent
        print("Found existing MorphFormer model:", model_file)
    else:
        print("Existing MorphFormer directory is incomplete; recloning it.")
        shutil.rmtree(external_dir / "morphFormer")
        repo_dir = external_dir / "morphFormer"

if model_file is None:
    subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(repo_dir)],
        check=True,
    )
    model_file = repo_dir / "model.py"

if not model_file.exists():
    # Give a diagnostic listing rather than failing silently on an unexpected layout.
    discovered = [str(p) for p in repo_dir.rglob("*.py")][:30]
    raise FileNotFoundError(
        f"Official MorphFormer model.py not found after checkout. "
        f"Repository directory: {repo_dir}. Python files found: {discovered}"
    )

# Put the directory containing the official model.py first on the import path.
sys.path.insert(0, str(model_file.parent))

import model as morph_official_model_module
from model import CNN

print("Official model file:", model_file)


Found existing MorphFormer model: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\external\morphFormer\morphFormer\model.py
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv3d-1       [-1, 8, 136, 11, 11]             656
       BatchNorm3d-2       [-1, 8, 136, 11, 11]              16
              ReLU-3       [-1, 8, 136, 11, 11]               0
            Conv2d-4           [-1, 64, 11, 11]          39,232
            Conv2d-5           [-1, 64, 11, 11]          69,696
           HetConv-6           [-1, 64, 11, 11]               0
       BatchNorm2d-7           [-1, 64, 11, 11]             128
              ReLU-8           [-1, 64, 11, 11]               0
           Dropout-9               [-1, 33, 64]               0
           Unfold-10              [-1, 288, 64]               0
        Erosion2d-11             [-1, 16, 8, 

In [5]:
# Model/data configuration.
PATCH_SIZE = 15
NUM_BANDS = 204
NUM_CLASSES = 16
FM = 16

# Official model.py uses a module-level patchsize in CNN.forward.
morph_official_model_module.patchsize = PATCH_SIZE

# Build datasets using the same spatial indices, then flatten each patch
# to match the official mainHouston.py input representation.
base_train_dataset = SalinasPatchDataset(
    scene=scene, indices=train_indices, patch_size=PATCH_SIZE, normalize=True
)
base_val_dataset = SalinasPatchDataset(
    scene=scene, indices=val_indices, patch_size=PATCH_SIZE, normalize=True
)
base_test_dataset = SalinasPatchDataset(
    scene=scene, indices=test_indices, patch_size=PATCH_SIZE, normalize=True
)

class MorphFormerSpatialDataset(Dataset):
    def __init__(self, base_dataset, expected_bands=NUM_BANDS, patch_size=PATCH_SIZE):
        self.base_dataset = base_dataset
        self.expected_bands = expected_bands
        self.patch_size = patch_size

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        patch, label = self.base_dataset[index]
        patch = np.asarray(patch, dtype=np.float32)

        # Project dataset yields [C, H, W]. Official MorphFormer receives
        # [C, H*W] and reshapes internally in CNN.forward.
        if patch.ndim != 3:
            raise ValueError(f"Expected patch [C,H,W], got {patch.shape}")
        if patch.shape[0] != self.expected_bands:
            raise ValueError(
                f"Expected {self.expected_bands} spectral bands, got {patch.shape[0]}"
            )
        if patch.shape[1:] != (self.patch_size, self.patch_size):
            raise ValueError(f"Expected spatial patch {(self.patch_size, self.patch_size)}, got {patch.shape[1:]}")

        patch_flat = np.ascontiguousarray(patch.reshape(self.expected_bands, -1))
        return torch.from_numpy(patch_flat), torch.tensor(int(label), dtype=torch.long)

morph_train_dataset = MorphFormerSpatialDataset(base_train_dataset)
morph_val_dataset = MorphFormerSpatialDataset(base_val_dataset)
morph_test_dataset = MorphFormerSpatialDataset(base_test_dataset)

print("Example input shape:", morph_train_dataset[0][0].shape)
print("Dataset sizes:", len(morph_train_dataset), len(morph_val_dataset), len(morph_test_dataset))

Example input shape: torch.Size([204, 225])
Dataset sizes: 32337 10952 10840


In [6]:
# Training and evaluation loaders.
TRAIN_BATCH_SIZE = 128
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

morph_train_loader = DataLoader(
    morph_train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
morph_val_loader = DataLoader(
    morph_val_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
morph_test_loader = DataLoader(
    morph_test_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

x_check, y_check = next(iter(morph_train_loader))
print("Batch input:", tuple(x_check.shape))  # expected [B, 204, 225]
print("Batch labels:", tuple(y_check.shape))

Batch input: (128, 204, 225)
Batch labels: (128,)


In [7]:
# The official CNN.forward() returns a 64-D feature vector (FM*4), not class logits.
# Its self.out3 layer is defined in model.py but is not called by forward().
# Wrap the official feature extractor with a trainable Salinas classification head.
class MorphFormerForClassification(nn.Module):
    def __init__(self, fm, num_bands, num_classes):
        super().__init__()
        self.backbone = CNN(
            FM=fm,
            NC=num_bands,
            Classes=num_classes,
            HSIOnly=True,
        )
        feature_dim = fm * 4
        # Reuse the classification layer already defined by the official CNN.
        # The official forward() does not invoke it, so call it explicitly here.
        self.classifier = self.backbone.out3

    def forward(self, x):
        features = self.backbone(x)
        expected_dim = self.classifier.in_features
        if features.ndim != 2 or features.shape[1] != expected_dim:
            raise RuntimeError(
                "Unexpected official MorphFormer feature shape: "
                f"{tuple(features.shape)}; expected [batch, {expected_dim}]"
            )
        return self.classifier(features)

morphformer_model = MorphFormerForClassification(
    fm=FM,
    num_bands=NUM_BANDS,
    num_classes=NUM_CLASSES,
).to(device)

morphformer_model.eval()
with torch.no_grad():
    check_logits = morphformer_model(x_check[:2].to(device, dtype=torch.float32))

print("Official MorphFormer feature dimension:", FM * 4)
print("Classifier output for 2 examples:", tuple(check_logits.shape))
assert tuple(check_logits.shape) == (2, NUM_CLASSES), (
    f"Expected output shape (2, {NUM_CLASSES}), got {tuple(check_logits.shape)}"
)

morphformer_param_count = sum(
    p.numel() for p in morphformer_model.parameters() if p.requires_grad
)
print(f"Trainable parameters: {morphformer_param_count:,}")
print(f"Parameters (M): {morphformer_param_count / 1e6:.6f}")


Official MorphFormer feature dimension: 64
Classifier output for 2 examples: (2, 16)
Trainable parameters: 280,752
Parameters (M): 0.280752


In [8]:
# Training setup follows the supplied MorphFormer notebook's optimizer family/settings.
MORPH_EPOCHS = 50
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 5e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    morphformer_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=50, gamma=0.9
)

In [9]:
def evaluate_morphformer(model, loader):
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=torch.float32)
            logits = model(x)
            pred = logits.argmax(dim=1).cpu().numpy()
            all_pred.append(pred)
            all_true.append(y.cpu().numpy())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(
        y_true, y_pred, labels=np.arange(NUM_CLASSES),
        average="macro", zero_division=0
    )
    return acc, macro_f1, y_true, y_pred

In [10]:
# Train using validation Macro-F1 for checkpoint selection.
best_val_macro_f1 = -1.0
best_epoch = 0
best_state = None
history = []

for epoch in range(1, MORPH_EPOCHS + 1):
    morphformer_model.train()
    running_loss = 0.0
    seen = 0

    for x, y in tqdm(morph_train_loader, desc=f"Epoch {epoch}/{MORPH_EPOCHS}", leave=False):
        x = x.to(device=device, dtype=torch.float32)
        y = y.to(device=device, dtype=torch.long)

        optimizer.zero_grad(set_to_none=True)
        logits = morphformer_model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        bs = y.size(0)
        running_loss += loss.item() * bs
        seen += bs

    train_loss = running_loss / max(seen, 1)
    val_acc, val_f1, _, _ = evaluate_morphformer(
        morphformer_model, morph_val_loader
    )
    scheduler.step()

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
    })

    if val_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_f1
        best_epoch = epoch
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in morphformer_model.state_dict().items()
        }

    print(
        f"Epoch {epoch:02d}/{MORPH_EPOCHS} | "
        f"loss={train_loss:.5f} | val_acc={val_acc:.5f} | "
        f"val_macro_f1={val_f1:.5f} | best_epoch={best_epoch}"
    )

if best_state is None:
    raise RuntimeError("Training did not produce a checkpoint state.")

morphformer_model.load_state_dict(best_state)
morphformer_model.to(device).eval()
print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", best_val_macro_f1)

Epoch 01/50 | loss=0.29524 | val_acc=0.91791 | val_macro_f1=0.84988 | best_epoch=1


Epoch 02/50 | loss=0.05608 | val_acc=0.93636 | val_macro_f1=0.93179 | best_epoch=2


Epoch 03/50 | loss=0.04751 | val_acc=0.91664 | val_macro_f1=0.86643 | best_epoch=2


Epoch 04/50 | loss=0.02589 | val_acc=0.92851 | val_macro_f1=0.93193 | best_epoch=4


Epoch 05/50 | loss=0.03379 | val_acc=0.90888 | val_macro_f1=0.90910 | best_epoch=4


...

Epoch 46/50 | loss=0.02227 | val_acc=0.89682 | val_macro_f1=0.92268 | best_epoch=8


Epoch 47/50 | loss=0.02157 | val_acc=0.92148 | val_macro_f1=0.93340 | best_epoch=8


Epoch 48/50 | loss=0.02345 | val_acc=0.86468 | val_macro_f1=0.87790 | best_epoch=8


Epoch 49/50 | loss=0.01411 | val_acc=0.91563 | val_macro_f1=0.93410 | best_epoch=8


Epoch 50/50 | loss=0.01601 | val_acc=0.89883 | val_macro_f1=0.94113 | best_epoch=8
Best epoch: 8
Best validation Macro-F1: 0.9613827362534617


In [11]:
# Save checkpoint and training history.
morphformer_checkpoint_path = outputs_dir / "morphformer_official_spatial_best.pt"
torch.save(
    {
        "model_state_dict": morphformer_model.state_dict(),
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "patch_size": PATCH_SIZE,
        "num_bands": NUM_BANDS,
        "num_classes": NUM_CLASSES,
        "FM": FM,
        "seed": SEED,
        "spatial_split": split_path.name,
        "official_repository": "https://github.com/mhaut/morphFormer",
    },
    morphformer_checkpoint_path,
)

history_path = outputs_dir / "morphformer_official_spatial_history.csv"
pd.DataFrame(history).to_csv(history_path, index=False)

print("Checkpoint:", morphformer_checkpoint_path)
print("Checkpoint exists:", morphformer_checkpoint_path.exists())
print("History:", history_path)

Checkpoint: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\morphformer_official_spatial_best.pt
Checkpoint exists: True
History: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\morphformer_official_spatial_history.csv


In [12]:
# Final held-out spatial test evaluation.
morph_accuracy, morph_macro_f1, y_true_morph, y_pred_morph = evaluate_morphformer(
    morphformer_model, morph_test_loader
)

print("=" * 60)
print("MORPHFORMER — SALINAS SPATIAL TEST RESULTS")
print("=" * 60)
print(f"Accuracy: {morph_accuracy:.6f}")
print(f"Macro-F1: {morph_macro_f1:.6f}")

print(classification_report(
    y_true_morph,
    y_pred_morph,
    labels=np.arange(NUM_CLASSES),
    target_names=[f"Class {i+1}" for i in range(NUM_CLASSES)],
    digits=4,
    zero_division=0,
))

MORPHFORMER — SALINAS SPATIAL TEST RESULTS
Accuracy: 0.939668
Macro-F1: 0.919630
              precision    recall  f1-score   support

     Class 1     0.9989    1.0000    0.9995       909
     Class 2     1.0000    0.6807    0.8100       498
     Class 3     0.9367    1.0000    0.9673       296
     Class 4     0.8576    1.0000    0.9233       295
     Class 5     1.0000    0.9432    0.9708       863
     Class 6     1.0000    1.0000    1.0000       632
     Class 7     1.0000    0.9917    0.9958       120
     Class 8     0.9864    0.8996    0.9410      2899
     Class 9     0.9983    1.0000    0.9992      1177
    Class 10     0.9408    0.7886    0.8580       544
    Class 11     0.3542    1.0000    0.5231        51
    Class 12     1.0000    1.0000    1.0000       448
    Class 13     1.0000    1.0000    1.0000       233
    Class 14     1.0000    0.9868    0.9934       228
    Class 15     0.7053    0.9467    0.8083       675
    Class 16     0.8594    1.0000    0.9244       972


In [13]:
# Save test predictions and confusion matrix.
morph_cm = confusion_matrix(
    y_true_morph, y_pred_morph, labels=np.arange(NUM_CLASSES)
)

pd.DataFrame({
    "y_true": y_true_morph,
    "y_pred": y_pred_morph,
}).to_csv(outputs_dir / "morphformer_official_spatial_predictions.csv", index=False)

pd.DataFrame(
    morph_cm,
    index=[f"Class {i+1}" for i in range(NUM_CLASSES)],
    columns=[f"Class {i+1}" for i in range(NUM_CLASSES)],
).to_csv(outputs_dir / "morphformer_official_spatial_confusion_matrix.csv")

pd.DataFrame([{
    "model": "MorphFormer (Official)",
    "accuracy": morph_accuracy,
    "macro_f1": morph_macro_f1,
    "best_epoch": best_epoch,
    "best_val_macro_f1": best_val_macro_f1,
}]).to_csv(outputs_dir / "morphformer_official_spatial_results.csv", index=False)

print("Saved spatial test outputs in:", outputs_dir)

Saved spatial test outputs in: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas


## Efficiency benchmark

The benchmark uses a dedicated batch of 128 test patches, 20 warm-up forward passes, and 100 timed forward passes. The benchmark input uses the same official flattened representation `[B, 204, 225]`.

THOP profiling is performed only on a deep copy. If THOP cannot profile the official morphology operators, FLOPs are recorded as `NaN`; latency benchmarking continues using the untouched model. GPU synchronization is conditional so this cell also runs on CPU.

In [14]:
# Fixed benchmark configuration and input batch.
BENCH_BATCH_SIZE = 128
WARMUP_RUNS = 20
TIMED_RUNS = 100

morph_bench_loader = DataLoader(
    morph_test_dataset,
    batch_size=BENCH_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

x_bench, y_bench = next(iter(morph_bench_loader))
x_bench = x_bench.to(device=device, dtype=torch.float32)
actual_batch_size = x_bench.shape[0]

print("Benchmark input shape:", tuple(x_bench.shape))
print("Actual batch size:", actual_batch_size)
assert x_bench.ndim == 3 and x_bench.shape[1:] == (NUM_BANDS, PATCH_SIZE * PATCH_SIZE)

Benchmark input shape: (128, 204, 225)
Actual batch size: 128


In [15]:
# Parameter count and saved checkpoint size.
morphformer_param_count = sum(
    p.numel() for p in morphformer_model.parameters() if p.requires_grad
)
morphformer_parameters_m = morphformer_param_count / 1e6

if not morphformer_checkpoint_path.exists():
    raise FileNotFoundError(f"Expected checkpoint missing: {morphformer_checkpoint_path}")

morphformer_model_size_mb = (
    morphformer_checkpoint_path.stat().st_size / (1024 ** 2)
)

print(f"Trainable parameters: {morphformer_param_count:,}")
print(f"Parameters (M): {morphformer_parameters_m:.6f}")
print(f"Checkpoint size (MB): {morphformer_model_size_mb:.6f}")

Trainable parameters: 280,752
Parameters (M): 0.280752
Checkpoint size (MB): 1.105522


In [16]:
# FLOPs on a deep copy ONLY; do not profile the benchmark model in place.
morphformer_flops = float("nan")
morphformer_gflops = float("nan")

try:
    from thop import profile

    flops_model = copy.deepcopy(morphformer_model).to(device).eval()
    dummy_input = torch.zeros(
        1, NUM_BANDS, PATCH_SIZE * PATCH_SIZE,
        device=device, dtype=torch.float32
    )

    with torch.no_grad():
        measured_flops, _ = profile(
            flops_model,
            inputs=(dummy_input,),
            verbose=False,
        )

    morphformer_flops = float(measured_flops)
    morphformer_gflops = morphformer_flops / 1e9
    print(f"FLOPs/sample: {morphformer_flops:,.0f}")
    print(f"GFLOPs/sample: {morphformer_gflops:.6f}")

except Exception as exc:
    print("THOP could not profile the official MorphFormer operators.")
    print(f"FLOPs will be saved as NaN. Reason: {type(exc).__name__}: {exc}")

finally:
    if "flops_model" in globals():
        del flops_model
    if "dummy_input" in globals():
        del dummy_input
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

FLOPs/sample: 66,584,352
GFLOPs/sample: 0.066584


In [17]:
# Peak allocated GPU memory during a batch-128 forward pass.
morphformer_model.eval()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

with torch.no_grad():
    _ = morphformer_model(x_bench)

if torch.cuda.is_available():
    torch.cuda.synchronize()
    morphformer_peak_memory_mb = (
        torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    )
else:
    morphformer_peak_memory_mb = float("nan")

print(f"Peak GPU memory (MB): {morphformer_peak_memory_mb:.4f}")

Peak GPU memory (MB): 504.8462


In [18]:
# Warm-up and 100 timed forward passes.
morphformer_model.eval()

with torch.no_grad():
    for _ in range(WARMUP_RUNS):
        _ = morphformer_model(x_bench)

if torch.cuda.is_available():
    torch.cuda.synchronize()

latencies_ms = []

with torch.no_grad():
    for _ in range(TIMED_RUNS):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        _ = morphformer_model(x_bench)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        latencies_ms.append((time.perf_counter() - t0) * 1000.0)

latencies_ms = np.asarray(latencies_ms, dtype=np.float64)

morphformer_mean_batch_ms = float(latencies_ms.mean())
morphformer_median_batch_ms = float(np.median(latencies_ms))
morphformer_p95_batch_ms = float(np.percentile(latencies_ms, 95))
morphformer_std_batch_ms = float(latencies_ms.std(ddof=1)) if len(latencies_ms) > 1 else 0.0
morphformer_latency_per_sample_ms = morphformer_mean_batch_ms / actual_batch_size
morphformer_throughput = actual_batch_size / (morphformer_mean_batch_ms / 1000.0)

print("=" * 60)
print("MORPHFORMER — EFFICIENCY BENCHMARK")
print("=" * 60)
print(f"Batch size: {actual_batch_size}")
print(f"Warm-up runs: {WARMUP_RUNS}")
print(f"Timed runs: {TIMED_RUNS}")
print(f"Mean batch latency: {morphformer_mean_batch_ms:.4f} ms")
print(f"Median batch latency: {morphformer_median_batch_ms:.4f} ms")
print(f"P95 batch latency: {morphformer_p95_batch_ms:.4f} ms")
print(f"Batch latency std: {morphformer_std_batch_ms:.4f} ms")
print(f"Latency/sample: {morphformer_latency_per_sample_ms:.6f} ms")
print(f"Throughput: {morphformer_throughput:.2f} samples/s")
print(f"Peak GPU memory: {morphformer_peak_memory_mb:.4f} MB")

MORPHFORMER — EFFICIENCY BENCHMARK
Batch size: 128
Warm-up runs: 20
Timed runs: 100
Mean batch latency: 211.0177 ms
Median batch latency: 179.0464 ms
P95 batch latency: 393.5186 ms
Batch latency std: 72.1699 ms
Latency/sample: 1.648576 ms
Throughput: 606.58 samples/s
Peak GPU memory: 504.8462 MB


In [19]:
# Final efficiency record with the same column names as the other model benchmarks.
morphformer_efficiency = pd.DataFrame([{
    "model": "MorphFormer (Official)",
    "accuracy": morph_accuracy,
    "macro_f1": morph_macro_f1,
    "parameters": morphformer_param_count,
    "parameters_m": morphformer_parameters_m,
    "model_size_mb": morphformer_model_size_mb,
    "flops": morphformer_flops,
    "gflops": morphformer_gflops,
    "peak_gpu_memory_mb": morphformer_peak_memory_mb,
    "batch_size": actual_batch_size,
    "latency_batch_ms": morphformer_mean_batch_ms,
    "latency_batch_median_ms": morphformer_median_batch_ms,
    "latency_batch_p95_ms": morphformer_p95_batch_ms,
    "latency_batch_std_ms": morphformer_std_batch_ms,
    "latency_per_sample_ms": morphformer_latency_per_sample_ms,
    "throughput_samples_sec": morphformer_throughput,
}])

display(morphformer_efficiency)

,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,MorphFormer (Official),0.939668,0.91963,280752,0.280752,1.105522,66584352.0,0.066584,504.846191,128,211.017723,179.0464,393.51862,72.169939,1.648576,606.584121


In [20]:
# Save and verify the efficiency CSV.
morphformer_efficiency_path = (
    outputs_dir / "morphformer_official_spatial_efficiency.csv"
)

morphformer_efficiency.to_csv(morphformer_efficiency_path, index=False)

print("Saved:", morphformer_efficiency_path.resolve())
print("Exists:", morphformer_efficiency_path.exists())

verified_efficiency = pd.read_csv(morphformer_efficiency_path)
display(verified_efficiency)

Saved: C:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\morphformer_official_spatial_efficiency.csv
Exists: True


,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,MorphFormer (Official),0.939668,0.91963,280752,0.280752,1.105522,66584352.0,0.066584,504.846191,128,211.017723,179.0464,393.51862,72.169939,1.648576,606.584121


In [21]:
# Optional compact summary of all saved MorphFormer spatial artifacts.
for path in sorted(outputs_dir.glob("morphformer_official_spatial*")):
    print(f"{path.name:55s} {path.stat().st_size:>10,} bytes")

morphformer_official_spatial_best.pt                     1,159,224 bytes
morphformer_official_spatial_confusion_matrix.csv              842 bytes
morphformer_official_spatial_efficiency.csv                    485 bytes
morphformer_official_spatial_history.csv                     3,146 bytes
morphformer_official_spatial_predictions.csv                59,909 bytes
morphformer_official_spatial_results.csv                       137 bytes
